# Clase 5 · Notebook 02
# Regresión múltiple y regularización (L1 / L2)

**Introducción al Deep Learning — Módulo 2, Unidad 3: Regresión**

Ampliamos la regresión a **varias variables** (regresión múltiple) con una red profunda y comprobamos cómo
la **regularización** ayuda a generalizar mejor.

1. Preparar los datos (todas las variables).
2. Red profunda para regresión múltiple.
3. Evaluar y comparar con la regresión simple.
4. Añadir regularización **L2** y comparar.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Preparar los datos (todas las variables)

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
tf.random.set_seed(42); np.random.seed(42)

datos = load_diabetes()
X = datos.data.astype("float32")     # 10 variables independientes
y = datos.target.astype("float32")
print("Atributos:", X.shape[1], "| Instancias:", X.shape[0])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. Red profunda para regresión múltiple

Varias capas ocultas **ReLU** y una salida **lineal** (un solo valor). Pérdida **MSE**, métrica **MAE**.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

def crear_modelo(regularizador=None):
    return Sequential([
        Input(shape=(X.shape[1],)),
        Dense(64, activation="relu", kernel_regularizer=regularizador),
        Dense(64, activation="relu", kernel_regularizer=regularizador),
        Dense(1,  activation="linear"),
    ])

modelo = crear_modelo()
modelo.compile(optimizer="adam", loss="mse", metrics=["mae"])
modelo.summary()

## 3. Entrenar y evaluar

In [ ]:
h = modelo.fit(X_train, y_train, epochs=150, batch_size=16,
               validation_split=0.1, verbose=0)
y_pred = modelo.predict(X_test, verbose=0).ravel()
print("Regresión múltiple (sin regularización)")
print(f"  R2 en test : {r2_score(y_test, y_pred):.3f}")
print(f"  MAE en test: {mean_absolute_error(y_test, y_pred):.1f}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(h.history["loss"], label="train", color="#000F9F")
plt.plot(h.history["val_loss"], label="validación", color="#FF647E")
plt.title("Pérdida (MSE) durante el entrenamiento")
plt.xlabel("Época"); plt.ylabel("MSE"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

La curva de validación que se separa de la de entrenamiento es señal de **sobreajuste**.
Aquí entra la regularización.

## 4. Añadir regularización L2

`kernel_regularizer = regularizers.l2(λ)` penaliza los pesos grandes para simplificar el modelo.
Comparamos el modelo con y sin regularización.

In [ ]:
from tensorflow.keras import regularizers

modelo_reg = crear_modelo(regularizador=regularizers.l2(0.01))
modelo_reg.compile(optimizer="adam", loss="mse", metrics=["mae"])
modelo_reg.fit(X_train, y_train, epochs=150, batch_size=16, validation_split=0.1, verbose=0)

y_pred_reg = modelo_reg.predict(X_test, verbose=0).ravel()
print("Comparación en el conjunto de test:\n")
print(f"  Sin regularización -> R2 = {r2_score(y_test, y_pred):.3f} | MAE = {mean_absolute_error(y_test, y_pred):.1f}")
print(f"  Con L2 (lambda=0.01) -> R2 = {r2_score(y_test, y_pred_reg):.3f} | MAE = {mean_absolute_error(y_test, y_pred_reg):.1f}")

La regularización L2 reduce el sobreajuste manteniendo (o mejorando) el rendimiento en datos nuevos.
También existe **L1** (`regularizers.l1`) y **Elastic Net** (`regularizers.l1_l2`).

## Experimenta tú

1. Cambia `l2(0.01)` por `l1(0.01)` o `l1_l2(0.01)` y compara el R².
2. Prueba distintos valores de λ (0.001, 0.1) y observa el efecto.
3. Reduce las épocas o las neuronas: ¿cómo cambia el sobreajuste?

## Resumen

- La **regresión múltiple** usa varias variables y capas ocultas: mejora el ajuste frente a la simple.
- La salida es **lineal** y la pérdida típica es **MSE**.
- La **regularización L2** penaliza los pesos grandes para **reducir el sobreajuste** y generalizar mejor.

Con esto cerramos el Módulo 2. En el **Módulo 3** veremos visión artificial con redes convolucionales.